In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(7, 7), dpi=150)
ax.set(xscale='log', yscale='log')
ax.set_xlabel(r'$\sin^2 2 \theta_{\mu \mu}$', fontsize=20)
ax.set_ylabel(r'$\Delta m_{41}^2$ [eV$^2$]', fontsize=20)

# === Define your inputs ===
file_list = [
    "TTnumu1_Surf0_surface.txt",
    "TTnumu1_Surf1_surface.txt",
    "TTnumu1_Surf2_surface.txt",
    "TTnumu1_Surf4_surface.txt",
    "TTnumu1_Surf5_surface.txt",
    "TTnumu1_Surf6_statonly_surface.txt"
]
#file_list = [
#    "TTnumu1_Surf0_surface.txt"
#]
labels = [
    r"$90\%$ CL (ICARUS) Asimov",
    r"No RPA CCQE",
    r"No NormCCMEC",
    r"No RPA_CCQE, NormCCMEC or Recomb",
    r"Only expskin_Flux, RPA_CCQE and Recomb",
    r"Stats Only"
]
#labels = [
#    r"$90\%$ CL (SBN+ICARUS) Asimov"
#]
colors = ['black','red','blue','lightgreen','orange','purple']

legend_proxies = []

# === Loop over files ===
for file, label, color in zip(file_list, labels, colors):
    # Detect header
    with open(file) as f:
        for i, line in enumerate(f):
            if line.strip().startswith("xval"):
                data_start = i + 1
                headers = line.strip().split()
                break

    # Read data
    data = np.loadtxt(file, skiprows=data_start)
    df = pd.DataFrame(data, columns=headers)

    # Extract and prepare grid
    x, y, z = df['xval'], df['yval'], df['chi2']
    min_chi = z.min()
    sig = df.pivot(index='xval', columns='yval', values='chi2').to_numpy()

    xi = np.linspace(x.min(), x.max(), len(np.unique(x)))
    yi = np.linspace(y.min(), y.max(), len(np.unique(y)))
    interp = RegularGridInterpolator((xi, yi), sig - min_chi, method='pchip')

    # Interpolation mesh
    N = 125
    xg = np.linspace(xi.min(), xi.max(), N)
    yg = np.linspace(yi.min(), yi.max(), N)
    xg_mesh, yg_mesh = np.meshgrid(xg, yg)
    zg = interp((xg_mesh, yg_mesh))

    # Plot contour at Δχ² = 1.6
    level = [1.6]
    ax.contour(10**xg, 10**yg, zg, levels=level, colors=[color], linewidths=2)

    # Save proxy for legend
    legend_proxies.append(Line2D([0], [0], color=color, lw=2))

# === Finalize plot ===
ax.legend(legend_proxies, labels, fontsize=16, loc='lower left').get_frame().set_linewidth(0.0)
ax.set_xlim(1e-2, 1)
ax.set_ylim(1e-2, 100)
plt.tight_layout()
plt.show()

